In [5]:
pip install agno

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.9 MB/s eta 0:00:00


In [7]:
pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.2 MB/s eta 0:00:00


In [9]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from ddgs import DDGS
from agno.workflow import Workflow

console = Console()

def chat_loop():

    search_agent = Agent(
        model=OpenAIChat(
            id="ai-sage/GigaChat3-10B-A1.8B",
            api_key="",
            base_url=""
        ),
        tools=[DDGS()],
        instructions=[
            "Ты полезный ассистент",
            "Используй инструменты поиска DuckDuckGo информации",
            "Отвечай на русском языке кратко и по существу"
        ],
        markdown=True
    )

    workflow = Workflow(
        name="GigaChat Search",
        steps=[search_agent]
    )

    console.print(Panel(
        "[bold cyan]GigaChat Search Assistant[/bold cyan]\n\n"
        "Команда [yellow]exit[/yellow] для выхода",
        title="Agno Workflow",
        border_style="cyan"
    ))

    while True:
        user_input = console.input("\n[bold green]Вы:[/bold green] ")

        if user_input.strip() == "":
            continue

        if user_input.strip().lower() == "exit":
            console.print(Panel(
                "[yellow]Покеда![/yellow]",
                border_style="yellow"
            ))
            break


        with console.status("[bold cyan]Обработка...[/bold cyan]", spinner="dots"):
            response = workflow.run(user_input)

        md = Markdown(response.content)
        console.print(Panel(
            md,
            title="[bold blue]GigaChat[/bold blue]",
            border_style="blue",
            padding=(1, 2)
        ))




if __name__ == "__main__":
    chat_loop()

╭───────────────────────────────────────────────── Agno Workflow ─────────────────────────────────────────────────╮
│ GigaChat Search Assistant                                                                                       │
│                                                                                                                 │
│ Команда exit для выхода                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Вы:

найди реальное имя инстасамки


Output()

╭─────────────────────────────────────────────────── GigaChat ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Реальное имя Насти Бужор, INSTASAMKA — её сценический псевдоним. Под этим именем она выступает как певица и    │
│  блогер.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Вы:

exit


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Покеда!                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
import uuid

console = Console()

def chat_loop():

    session_id = str(uuid.uuid4())

    search_agent = Agent(
        model=OpenAIChat(
            id="ai-sage/GigaChat3-10B-A1.8B",
            api_key="",
            base_url=""
        ),

        tools=[DuckDuckGoTools()],

        instructions=[
            """
            Ты умный поисковый ассистент с памятью и планированием.
            PLANNING:
            - Перед выполнением задачи анализируй запрос,
            - Составь план действий: что нужно найти, как структурировать ответ,
            - Разбивай сложные запросы на подзадачи,

            MEMORY:
            - Запоминай предыдущие вопросы пользователя в этой сессии,
            - Используй контекст предыдущих сообщений для понимания,
            - Если пользователь ссылается на ранее обсужденное - учитывай это,

            KNOWLEDGE:
            - Используй свои базовые знания,
            - Дополняй их актуальной информацией из DuckDuckGo,
            - Синтезируй информацию из разных источников,

            Используй DuckDuckGo для поиска актуальной информации,
            Отвечай на русском языке структурированно и по существу
            """
        ],

        markdown=True,
    )

    console.print(Panel(
        "[bold cyan]Smart Agent[/bold cyan]\n\n"
        "PLANNING: Анализ и планирование через промпт\n"
        "MEMORY: Контекст в рамках сессии\n"
        "KNOWLEDGE: Базовые знания + DuckDuckGo\n\n"
        f"Session: [dim]{session_id[:8]}...[/dim]\n\n"
        "Команда [yellow]exit[/yellow] для выхода",
        title="Agno Agent",
        border_style="cyan"
    ))

    conversation_history = []
    message_count = 0

    while True:
        user_input = console.input("\n[bold green]Вы:[/bold green] ")

        if user_input.strip() == "":
            continue

        if user_input.strip().lower() == "exit":
            console.print(Panel(
                f"[yellow]Cпасибо деду и покеда![/yellow]",
                border_style="yellow"
            ))
            break

        message_count += 1

        if conversation_history:
            context = "\n".join([
                f"[Предыдущий вопрос {i+1}]: {msg['q']}\n[Ответ]: {msg['a'][:200]}..."
                for i, msg in enumerate(conversation_history[-3:])
            ])
            full_prompt = f"{context}\n\n[Текущий вопрос]: {user_input}"
        else:
            full_prompt = user_input

        with console.status("[bold cyan]Processing...[/bold cyan]", spinner="dots"):
            response = search_agent.run(full_prompt)

        conversation_history.append({
            'q': user_input,
            'a': response.content
        })

        md = Markdown(response.content)
        console.print(Panel(
            md,
            title=f"[bold blue]GigaChat[/bold blue]",
            border_style="blue",
            padding=(1, 2)
        ))


if __name__ == "__main__":
    chat_loop()

╭────────────────────────────────────────────────── Agno Agent ───────────────────────────────────────────────────╮
│ Smart Agent                                                                                                     │
│                                                                                                                 │
│ PLANNING: Анализ и планирование через промпт                                                                    │
│ MEMORY: Контекст в рамках сессии                                                                                │
│ KNOWLEDGE: Базовые знания + DuckDuckGo                                                                          │
│                                                                                                                 │
│ Session: 7fd5f29b...                                                                                            │
│                                                                                                                 │
│ Команда exit для выхода                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Вы:

запомни, что я пью только фильтр кения


Output()

╭─────────────────────────────────────────────────── GigaChat ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Вы уже знаете из предыдущих сообщений, что я запомнила про кенийский фильтр. Это хороший выбор - кенийский     │
│  кофе известен своим насыщенным вкусом с нотами ягод и косточковых фруктов.                                     │
│                                                                                                                 │
│  Для подготовки кенийского фильтра я бы предложил следующую стратегию:                                          │
│                                                                                                                 │
│   1 Сначала проверю свежие рекомендации по приготовлению кенийского кофе                                        │
│   2 Затем сосредоточусь на дополнительных деталях, которые делают кенийский кофе особенным                      │
│   3 Уточню, как другие пользователи оценивают кенийский фильтр для аэропресса                                   │
│   4 Поделюсь наиболее важными моментами для приготовления                                                       │
│                                                                                                                 │
│  Если вам нужна дополнительная информация или вы хотите что-то уточнить, я рада помочь!                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭─────────────────────────────────────────────────── GigaChat ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Вы пили кенийский фильтр-кофе. Запомните этот сорт — это хороший выбор с насыщенным вкусом и нотами ягод и     │
│  косточковых фруктов.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Вы:

exit


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Cпасибо деду и покеда!                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
import uuid

console = Console()


def create_agents():

    consultant = Agent(
        name="Консультант по гопникам",
        model=OpenAIChat(
            id="ai-sage/GigaChat3-10B-A1.8B",
            api_key="",
            base_url=""
        ),

        tools=[DuckDuckGoTools()],

        instructions=[
            """
            Ты - КОНСУЛЬТАНТ ПО ГОПНИКАМ на съемках сериала 'Реальные пацаны',
            Твоя роль: эксперт по дворовой субкультуре 90-х и 2000-х,
            ТВОЯ ЭКСПЕРТИЗА:
            - Знаешь все жаргонные слова: 'кореш', 'братан', 'замутить', 'в натуре', 'по ходу',
            - Понимаешь манеры поведения: расслабленная походка, руки в карманах, прищур,
            - Разбираешься в дресс-коде: спортивки Adidas, Nike, кепки, золотые цепи,
            - Знаешь места обитания: подъезды, дворы, остановки, гаражи,
            - Понимаешь иерархию: кто 'авторитет', кто 'шестёрка', кто 'пацан',

            ТВОЯ ЗАДАЧА:
            - Найди информацию о том, что запросили (через DuckDuckGo если нужно),
            - Опиши детали дворовой жизни максимально authentically,
            - Дай конкретные примеры фраз, жестов, поведения,
            - Расскажи про атмосферу и контекст,

            СТИЛЬ ОТВЕТА:
            Пиши профессионально, но с пониманием темы,
            Приводи реальные примеры из жизни,
            Будь максимально детальным - это важно для актёров!
            """
        ],

        markdown=True,
    )

    screenwriter = Agent(
        name="Сценарист",
        model=OpenAIChat(
            id="ai-sage/GigaChat3-10B-A1.8B",
            api_key="",
            base_url=""
        ),

        tools=[],

        instructions=[
            """
            Ты - СЦЕНАРИСТ сериала 'Реальные пацаны',
            Твоя роль: писать живые диалоги и сцены про дворовых пацанов,

            ТВОИ ЗАДАЧИ:
            - Получи информацию от консультанта по гопникам",
            - Создай диалоги на основе реального жаргона",
            - Напиши короткую сцену (1-2 минуты экранного времени)",
            - Пропиши ремарки для актёров: жесты, интонации, мимика",

            ФОРМАТ СЦЕНАРИЯ:
            СЦЕНА: [описание места и времени],
            ДЕЙСТВИЕ:[описание что происходит]
            ДИАЛОГ:
            ПЕРСОНАЖ 1 (ремарка - как говорить): Реплика,
            ПЕРСОНАЖ 2 (ремарка): Реплика,
            ВАЖНО:
            - Диалоги должны звучать естественно,
            - Используй жаргон, но не переборщи,
            - Добавь атмосферу через детали,
            - Помни про юмор - сериал комедийный!
            """
        ],

        markdown=True
    )

    actor = Agent(
        name="Актёр",
        model=OpenAIChat(
            id="ai-sage/GigaChat3-10B-A1.8B",
            api_key="",
            base_url=""
        ),

        tools=[],

        instructions=[
            """
            Ты - АКТЁР, играющий гопника в сериале 'Реальные пацаны',
            Твоя роль: подготовиться к съёмке сцены,

            ТВОИ ЗАДАЧИ:
            - Изучи информацию от консультанта,
            - Прочитай сценарий от сценариста,
            - Создай АКТЁРСКУЮ ШПАРГАЛКУ для съёмки,

            ФОРМАТ ШПАРГАЛКИ:

            1. КЛЮЧЕВЫЕ ФРАЗЫ,
            [список фраз которые буду использовать],

            2. МАНЕРА РЕЧИ,
            [как говорить: темп, интонация, паузы],

            3. ЖЕСТЫ И ДВИЖЕНИЯ,
            [что делать руками, как стоять, куда смотреть],

            4. МИМИКА И ЭМОЦИИ,
            [выражение лица, эмоции],

            5. ОБРАЗ ПЕРСОНАЖА,
            [кто мой персонаж, его характер, мотивация],

            6. ВАЖНЫЕ МОМЕНТЫ СЦЕНЫ,
            [на что обратить внимание при игре],

            СТИЛЬ:
            - Пиши кратко и по делу - это рабочие заметки,
            - Добавь свои актёрские наблюдения,
            - Будь готов к импровизации
            """
        ],

        markdown=True
    )

    return consultant, screenwriter, actor

def film_crew(request, consultant, screenwriter, actor):

    console.print("\n[bold cyan]СЪЁМОЧНАЯ ГРУППА 'РЕАЛЬНЫЕ ПАЦАНЫ'[/bold cyan]\n")

    with console.status("[cyan]Сбор информации о дворовой культуре...[/cyan]", spinner="dots"):
        consultant_info = consultant.run(
            f"Дай детальную информацию для съёмочной группы:\n{request}\n\n"
            f"Нужны: конкретные слова, манеры, жесты, атмосфера."
        )

    with console.status("[cyan]Работа над диалогами...[/cyan]", spinner="dots"):
        script = screenwriter.run(
            f"На основе информации от консультанта напиши сцену:\n\n"
            f"ЗАПРОС: {request}\n\n"
            f"МАТЕРИАЛЫ ОТ КОНСУЛЬТАНТА:\n{consultant_info.content}\n\n"
            f"Создай короткую сцену (1-2 минуты) с диалогами и ремарками."
        )

    console.print("[yellow]Актёр готовится к съёмке...[/yellow]")
    with console.status("[cyan]Изучение роли...[/cyan]", spinner="dots"):
        actor_notes = actor.run(
            f"Изучи материалы и подготовься к съёмке:\n\n"
            f"ЧТО НУЖНО СЫГРАТЬ:\n{request}\n\n"
            f"ИНФОРМАЦИЯ ОТ КОНСУЛЬТАНТА:\n{consultant_info.content[:800]}...\n\n"
            f"СЦЕНАРИЙ:\n{script.content}\n\n"
            f"Создай актёрскую шпаргалку для съёмки."
        )

    return actor_notes.content


def chat_loop():

    consultant, screenwriter, actor = create_agents()

    session_id = str(uuid.uuid4())

    console.print(Panel(
        "[bold cyan]Съёмочная группа сериала 'Реальные пацаны'[/bold cyan]\n\n"
        "[yellow]Консультант по гопникам[/yellow] - эксперт по дворовой культуре\n"
        "[blue]Сценарист[/blue] - создаёт диалоги и сцены\n"
        "[green]Актёр[/green] - готовится к роли\n\n"
        f"Session: [dim]{session_id[:8]}...[/dim]\n\n"
        "Команда [yellow]exit[/yellow] для завершения",
        title="Agno Film Crew",
        border_style="cyan"
    ))

    scene_count = 0

    while True:
        user_input = console.input("\n[bold green]Режиссёр:[/bold green] ")

        if user_input.strip() == "":
            continue

        if user_input.strip().lower() == "exit":
            console.print(Panel(
                f"[yellow]Съёмочный день завершён.[/yellow]\n"
                f"Спасибо за работу!",
                border_style="yellow"
            ))
            break

        scene_count += 1

        final_result = film_crew(user_input, consultant, screenwriter, actor)

        md = Markdown(final_result)
        console.print(Panel(
            md,
            title=f"[bold magenta]Актёр готов![/bold magenta]",
            border_style="magenta",
            padding=(1, 2)
        ))


if __name__ == "__main__":
    chat_loop()


╭──────────────────────────────────────────────── Agno Film Crew ─────────────────────────────────────────────────╮
│ Съёмочная группа сериала 'Реальные пацаны'                                                                      │
│                                                                                                                 │
│ Консультант по гопникам - эксперт по дворовой культуре                                                          │
│ Сценарист - создаёт диалоги и сцены                                                                             │
│ Актёр - готовится к роли                                                                                        │
│                                                                                                                 │
│ Session: e0b66926...                                                                                            │
│                                                                                                                 │
│ Команда exit для завершения                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Режиссёр:

Как приветствуют друг друга реальные пацаны?


СЪЁМОЧНАЯ ГРУППА 'РЕАЛЬНЫЕ ПАЦАНЫ'

Output()

...

Output()

╭───────────────────────────────────────────────── Актёр готов! ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓  │
│  ┃                                 АКТЁРСКАЯ ШПАРГАЛКА: ИСТОРИЯ К КУЧЕРЯВОМУ                                 ┃  │
│  ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛  │
│                                                                                                                 │
│                                                                                                                 │
│                                          1. ДУБЛИНКА ЗА ОКНОМ ОТСНЯТА                                           │
│                                                                                                                 │
│                                              Мотивация характера:                                               │
│                                                                                                                 │
│   • Ликвидирую свою девушку, заливаясь улыбкой пола. Началось это, когда она начала её фантазировать: туалет    │
│     уборщицы в костюмах для контакта новичков.                                                                  │
│                                                                                                                 │
│                                                    ХАРАКТЕР:                                                    │
│                                                                                                                 │
│   • Наглость и брутальность с юмором наваливается быстро                                                        │
│   • Оправдание через ложь                                                                                       │
│   • Финансы твои и моя ваша выгода усиливают мою низкую самооценку                                              │
│                                                                                                                 │
│                                                    ИНТЕРЕСЫ:                                                    │
│                                                                                                                 │
│   • Пользоваться всеми удобствами понедельника                                                                  │
│   • Получать комплименты за тренды                                                                              │
│                                                                                                                 │
│                                                    ЦЕННОСТИ:                                                    │
│                                                                                                                 │
│   • Мой пропускает довериться мой ветер                                                                         │
│   • Мой толк о моей щедрости и любовь природе нашей Семейной Тойоты-салют                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                         2. ВРАГИ: НАЗВАНИЕ БУТИБЕЛЬНЫЕ                                          │
│                                                                                                                 │
│                                             Роль мотивов нежелания:                                             │
│                                                       

Режиссёр:

exit


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Съёмочный день завершён.                                                                                        │
│ Спасибо за работу!                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯